# Repo Architecture Walkthrough

This notebook is the index for the current walkthrough set. It focuses on
where code lives today and how the repo is intentionally split.

Walkthrough set:

1. `00_repo_architecture_walkthrough.ipynb`
2. `01_db_and_migrations_walkthrough.ipynb`
3. `02_importer_walkthrough.ipynb`
4. `03_inventory_domain_walkthrough.ipynb`
5. `04_reporting_and_api_contract_walkthrough.ipynb`

The goal is not to mirror every file. It is to orient you to the stable
runtime boundaries that the rest of the notebooks build on.


## Setup

This notebook is light on runtime state. The setup cell just finds the repo,
imports the local package directly from `src/`, and defines a few display
helpers.


In [ ]:
from __future__ import annotations

import importlib
import inspect
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mtg_source_stack').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
PACKAGE_ROOT = SRC_DIR / 'mtg_source_stack'

SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

from mtg_source_stack.inventory.report_helpers import render_table


def walk_package(root: Path, *, max_depth: int = 2) -> list[str]:
    lines: list[str] = []

    def visit(path: Path, depth: int) -> None:
        if depth > max_depth:
            return
        for child in sorted(path.iterdir()):
            if child.name == '__pycache__':
                continue
            prefix = '  ' * depth
            suffix = '/' if child.is_dir() else ''
            lines.append(f'{prefix}{child.name}{suffix}')
            if child.is_dir():
                visit(child, depth + 1)

    visit(root, 0)
    return lines


def module_row(module_name: str) -> dict[str, str]:
    module = importlib.import_module(module_name)
    doc = inspect.getdoc(module) or ''
    first_line = doc.splitlines()[0] if doc else '(no module docstring)'
    return {
        'module': module_name.replace('mtg_source_stack.', ''),
        'path': str(Path(module.__file__).resolve().relative_to(REPO_ROOT)),
        'purpose': first_line,
    }


print('repo root:', REPO_ROOT)
print('package root:', PACKAGE_ROOT)


## Current Package Map

The package tree below matches the current `db / importer / inventory / cli`
shape described in `docs/architecture.md`.


In [ ]:
print('\n'.join(walk_package(PACKAGE_ROOT, max_depth=2)))


## Core Docs and Public Surfaces

These are the docs and modules that matter most when you are orienting to
the runtime architecture rather than to a single feature.


In [ ]:
core_rows = [
    {'surface': 'docs/architecture.md', 'role': 'High-level runtime package map'},
    {'surface': 'docs/backend_v1_contract.md', 'role': 'Canonical backend/schema rules'},
    {'surface': 'docs/api_v1_contract.md', 'role': 'HTTP-facing JSON and error contract'},
    {'surface': 'src/mtg_source_stack/inventory/service.py', 'role': 'Intentional public inventory facade'},
    {'surface': 'src/mtg_source_stack/personal_inventory_cli.py', 'role': 'Legacy inventory wrapper entrypoint'},
    {'surface': 'src/mtg_source_stack/mvp_importer.py', 'role': 'Legacy importer wrapper entrypoint'},
]

print(render_table(core_rows, [('surface', 'surface'), ('role', 'role')]))


## Inventory Package Shape

The inventory package is the most important application boundary. The rows
below show the current core modules and their intended jobs.


In [ ]:
module_names = [
    'mtg_source_stack.inventory.service',
    'mtg_source_stack.inventory.inventories',
    'mtg_source_stack.inventory.catalog',
    'mtg_source_stack.inventory.mutations',
    'mtg_source_stack.inventory.analysis',
    'mtg_source_stack.inventory.csv_import',
    'mtg_source_stack.inventory.audit',
    'mtg_source_stack.inventory.response_models',
    'mtg_source_stack.inventory.policies',
    'mtg_source_stack.inventory.query_catalog',
    'mtg_source_stack.inventory.query_inventory',
    'mtg_source_stack.inventory.query_pricing',
    'mtg_source_stack.inventory.query_reporting',
    'mtg_source_stack.inventory.report_formatters',
    'mtg_source_stack.inventory.report_helpers',
    'mtg_source_stack.inventory.report_io',
]

rows = [module_row(name) for name in module_names]
print(render_table(rows, [('module', 'module'), ('path', 'path'), ('purpose', 'purpose')]))


## Suggested Reading Order

If you are new to the repo, a good path is:

1. `docs/architecture.md`
2. `docs/backend_v1_contract.md`
3. `inventory/service.py`
4. `db/` and `importer/`
5. the remaining notebooks in this directory

The next notebook starts with the persistence layer so the later importer and
inventory notebooks can assume the schema and migration story are already in
focus.
